# MCP Evaluation Metrics Notebook


## 1. Install dependencies


In [ ]:
%pip install \
  openai \
  pandas \
  openpyxl


## 2. Import libraries


In [ ]:
import os
import json
from pathlib import Path

import pandas as pd
from openai import AzureOpenAI


## 3. Set your Azure OpenAI credentials


In [ ]:
os.environ["AZURE_OPENAI_API_KEY"] = "YOUR_AZURE_OPENAI_API_KEY"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://YOUR-RESOURCE-NAME.openai.azure.com/"
os.environ["AZURE_OPENAI_API_VERSION"] = "2024-02-01"
os.environ["AZURE_OPENAI_DEPLOYMENT"] = "YOUR_DEPLOYMENT_NAME"


In [ ]:
api_key = os.environ.get("AZURE_OPENAI_API_KEY")
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
api_version = os.environ.get("AZURE_OPENAI_API_VERSION")
deployment = os.environ.get("AZURE_OPENAI_DEPLOYMENT")

print("API key loaded:  ", bool(api_key))
print("Endpoint loaded: ", bool(endpoint))
print("API version:     ", api_version)
print("Deployment:      ", deployment)


## 4. Load the MCP evaluation input file


In [ ]:
DATA_FILE = Path(r"C:\Arghya\Work\Colab Notebook Version Upgrades\New Notebooks\MCP_Evaluation_Test_Data.xlsx")

client = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=endpoint,
)

df = pd.read_excel(DATA_FILE)
df


## 5. Define the MCP evaluation metrics


In [ ]:
MCP_METRICS = [
    {
        "name": "MCP Format Check",
        "short_name": "protocol_compliance",
        "what_it_checks": "Checks whether the actual output follows MCP-style structure, stays within MCP scope, and avoids agent-specific behavior.",
        "judge_points": [
            "Does the output look compatible with MCP concepts such as tools, resources, schemas, URIs, or structured server responses?",
            "Does it stay focused on MCP interaction patterns instead of agent planning, memory, or autonomy?",
            "Is the structure consistent, predictable, and safe for a client to consume?"
        ]
    },
    {
        "name": "MCP Tool Clarity",
        "short_name": "tool_resource_clarity",
        "what_it_checks": "Checks whether the tool or resource design is clear, well named, and easy for an MCP client to understand.",
        "judge_points": [
            "Are tool names, resource names, descriptions, and parameters clear and purposeful?",
            "Would a developer understand how to call the tool or use the resource without extra guessing?",
            "Are the inputs and outputs described in a way that reduces ambiguity?"
        ]
    },
    {
        "name": "MCP Integration Readiness",
        "short_name": "interop_dx",
        "what_it_checks": "Checks whether the output supports reuse across clients and gives developers a smooth integration experience.",
        "judge_points": [
            "Would different MCP clients be able to use this output consistently?",
            "Does it encourage stable contracts such as schemas, field names, MIME types, and predictable JSON shapes?",
            "Does it make implementation easier for developers through clarity and practical detail?"
        ]
    }
]

for metric in MCP_METRICS:
    print(metric["name"])
    print("-", metric["what_it_checks"])
    print()


## 6. Build the MCP judge prompt and evaluation helper


In [ ]:
def build_judge_prompt(metric_name, metric_description, judge_points, user_input, actual_output, expected_output):
    rubric = "
".join([f"- {point}" for point in judge_points])
    return f"""You are evaluating an output for an MCP-only use case.

Metric name: {metric_name}
Metric description: {metric_description}

Important boundary:
- Evaluate only for Model Context Protocol (MCP).
- Do not reward or penalize agent behaviors such as planning, autonomy, memory, orchestration, or multi-step task execution unless they directly create MCP confusion.
- Focus only on MCP concerns like protocol shape, tools, resources, schemas, interoperability, and developer usability.

Scoring rubric:
{rubric}

Evaluation input:
Input: {user_input}
Actual output: {actual_output}
Expected output: {expected_output}

Return strict JSON with this shape:
{{
  "score": <number from 1 to 5>,
  "reason": "short explanation"
}}
"""


def evaluate_with_llm(metric, user_input, actual_output, expected_output):
    prompt = build_judge_prompt(
        metric_name=metric["name"],
        metric_description=metric["what_it_checks"],
        judge_points=metric["judge_points"],
        user_input=user_input,
        actual_output=actual_output,
        expected_output=expected_output,
    )

    response = client.chat.completions.create(
        model=deployment,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "You are a precise evaluator for MCP outputs."},
            {"role": "user", "content": prompt},
        ],
    )

    content = response.choices[0].message.content
    parsed = json.loads(content)
    return {
        "score": parsed.get("score"),
        "reason": parsed.get("reason", "")
    }


## 7. Run the three MCP evaluations


In [ ]:
results = []

for _, row in df.iterrows():
    for metric in MCP_METRICS:
        evaluation = evaluate_with_llm(
            metric=metric,
            user_input=row["input"],
            actual_output=row["actual_output"],
            expected_output=row["expected_output"],
        )
        results.append({
            "scenario_id": row["scenario_id"],
            "metric": metric["name"],
            "score": evaluation["score"],
            "reason": evaluation["reason"],
        })

results_df = pd.DataFrame(results)
results_df


## 8. Review the metric summary


In [ ]:
summary_df = (
    results_df.groupby("metric", as_index=False)["score"]
    .mean()
    .rename(columns={"score": "average_score"})
    .sort_values("average_score", ascending=False)
)

summary_df


## 9. Save the evaluation results


In [ ]:
output_file = Path(r"C:\Arghya\Work\Colab Notebook Version Upgrades\New Notebooks\MCP_Evaluation_Results.xlsx")

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="input_data")
    results_df.to_excel(writer, index=False, sheet_name="metric_results")
    summary_df.to_excel(writer, index=False, sheet_name="summary")

print(f"Saved evaluation results to: {output_file}")


## Recap


In [ ]:
print("Notebook structure followed:")
print("1. Installations")
print("2. Imports")
print("3. Credentials")
print("4. Main MCP evaluation flow")
print()
print("Input Excel file:", DATA_FILE)
print("Result Excel file:", output_file)
